# RecScale: Real-Time Event-Driven Recommendation & Ranking Engine
### End-to-End Demonstration on Free Google Colab

This interactive notebook demonstrates the complete multi-stage recommendation pipeline:
1. **Real-Time Clickstream Ingestion:** Replaying real MovieLens events into the online feature store.
2. **Two-Tower Neural Retrieval:** Sub-5ms Candidate generation via FAISS HNSW index (Top-500).
3. **Deep & Cross Network v2 (DCNv2) Real-Time Ranking:** Explicit high-degree polynomial feature interactions for CTR prediction.
4. **Maximal Marginal Relevance (MMR) Diversification:** Calibrated top-20 feed preventing filter bubbles.
5. **Real-Time Data & Concept Drift Monitoring:** Population Stability Index (PSI) and KS-drift testing.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch numpy scipy fastapi uvicorn prometheus-client pydantic faiss-cpu

In [ ]:
# Step 2: Download Real-World MovieLens Datasets (Ratings & Catalog)
!python scripts/download_recsys_data.py

In [ ]:
# Step 3: Run Core Unit Tests
!python tests/test_two_tower.py
!python tests/test_dcn_v2.py
!python tests/test_hnsw.py
!python tests/test_mmr.py
!python tests/test_streaming.py

In [ ]:
# Step 4: Interactive Recommendation Demo
from rec_engine.pipeline import RecommendationPipeline
from rec_engine.streaming.event_schemas import UserClickEvent

pipeline = RecommendationPipeline()
pipeline.bootstrap_catalog("data_recsys/movies.csv")

user_id = 42
# Simulate streaming click events (User is clicking on Sci-Fi / Action movies)
print("=== Simulating Streaming Click Events ===")
for item_id in [1, 260, 1196, 1210]: # Toy Story, Star Wars, Empire Strikes Back
    event = UserClickEvent(user_id=user_id, item_id=item_id, category="Sci-Fi")
    pipeline.aggregator.process_click_event(event)
    print(f"[*] Click Event Logged: User {user_id} clicked Item {item_id}")

# Execute Multi-Stage Recommendation
cands, _, ret_lat = pipeline.retriever.retrieve_candidates(user_id, top_k=500)
ranked_ids, scores, rank_lat = pipeline.ranker.rank_candidates(user_id, cands)
div_ids, div_scores, mmr_lat = pipeline.diversifier.diversify(ranked_ids, scores, top_k=5)

print(f"\n=== Top-5 Personalized Recommendations (Latencies: Ret={ret_lat:.2f}ms, Rank={rank_lat:.2f}ms, MMR={mmr_lat:.2f}ms) ===")
for idx, (iid, s) in enumerate(zip(div_ids, div_scores)):
    meta = pipeline.feature_store.get_item_features(iid)
    print(f" #{idx+1} [CTR: {s:.4f}] {meta.get('title', 'Unknown')} ({meta.get('category')})")

In [ ]:
# Step 5: Run Full Production Latency Profiler Benchmark
!python benchmarks/benchmark_recsys.py